# OntologyQA - OpenRouter Smoke Test

Notebook này kiểm tra nhanh việc gọi OpenRouter cho baseline **1.1 LLM-only** bằng 1 mẫu từ `test_questions_v1.0.xlsx`.

Mục tiêu của notebook:
- Đọc 1 câu hỏi trắc nghiệm hợp lệ từ file Excel.
- Gọi OpenRouter qua `openai` SDK với model mặc định `google/gemma-4-26b-a4b-it:free`.
- Yêu cầu model chọn đáp án, không dùng SPARQL/ontology retrieval.
- In response headers để phục vụ bước đo latency/TPS ở các thí nghiệm sau.

## 1. Cấu hình môi trường

In [1]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

EXCEL_PATH = Path("test_questions_v1.0.xlsx")
ENV_PATH = Path(".env")
load_dotenv(ENV_PATH)

OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL = os.getenv("OPENROUTER_MODEL", "google/gemma-4-26b-a4b-it")
API_KEY = os.getenv("OPENROUTER_API_KEY")
SAMPLE_ID = 11

print(f"Excel exists: {EXCEL_PATH.exists()} -> {EXCEL_PATH.resolve()}")
print(f".env exists: {ENV_PATH.exists()} -> {ENV_PATH.resolve()}")
print(f"OpenRouter base URL: {OPENROUTER_BASE_URL}")
print(f"Model: {MODEL}")
print(f"Sample ID: {SAMPLE_ID}")
print(f"API key configured: {bool(API_KEY)}")

Excel exists: True -> D:\Dev\VDT2026-OntologyQA\test_questions_v1.0.xlsx
.env exists: True -> D:\Dev\VDT2026-OntologyQA\.env
OpenRouter base URL: https://openrouter.ai/api/v1
Model: google/gemma-4-26b-a4b-it
Sample ID: 11
API key configured: True


## 2. Đọc 1 mẫu hợp lệ từ Excel

File hiện có một số dòng trống/chưa hoàn thiện. Cell này chỉ lấy dòng có đủ câu hỏi, chỉ số đáp án đúng và ít nhất 2 option.

In [2]:
def none_if_nan(value):
    return None if pd.isna(value) else value


def normalize_question_record(record: dict) -> dict | None:
    question = none_if_nan(record.get("vi_question"))
    correct_option = none_if_nan(record.get("answer"))
    options = [none_if_nan(record.get(f"option_{idx}")) for idx in range(1, 6)]
    available_options = [option for option in options if option is not None]

    if not question or correct_option is None or len(available_options) < 2:
        return None

    return {
        "number": int(record.get("number")),
        "question": str(question),
        "question_type": str(none_if_nan(record.get("question_type")) or "").strip(),
        "gold_answer": none_if_nan(record.get("gold_answer")),
        "correct_option": int(correct_option),
        "options": options,
    }


def load_valid_questions(path: Path) -> list[dict]:
    df = pd.read_excel(path, engine="openpyxl")
    df = df.rename(columns={"Unnamed: 3": "gold_answer"})
    records = []

    for record in df.to_dict(orient="records"):
        normalized = normalize_question_record(record)
        if normalized is not None:
            records.append(normalized)

    return records


def get_question_by_id(path: Path, sample_id: int) -> dict:
    records = load_valid_questions(path)
    for record in records:
        if record["number"] == sample_id:
            return record

    available_ids = [record["number"] for record in records]
    raise ValueError(f"Không tìm thấy sample id {sample_id}. Các id hợp lệ: {available_ids}")


sample = get_question_by_id(EXCEL_PATH, SAMPLE_ID)
print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "number": 11,
  "question": "Nguyễn Hữu Cảnh có bao nhiêu người thân",
  "question_type": "counting",
  "gold_answer": 3,
  "correct_option": 3,
  "options": [
    1,
    2,
    3,
    4,
    5
  ]
}


## 3. Prompt baseline 1.1

Baseline 1.1 không gọi SPARQL, không retrieve ontology, không dùng dữ liệu ngoài. Model chỉ dựa vào parametric memory và các option đã cho.

In [3]:
def build_llm_only_messages(item: dict) -> list[dict[str, str]]:
    options_text = "\n".join(
        f"{idx}. {option}"
        for idx, option in enumerate(item["options"], start=1)
        if option is not None
    )

    user_prompt = f"""Câu hỏi tiếng Việt:
{item['question']}

Loại câu hỏi: {item['question_type'] or 'unknown'}

Các lựa chọn:
{options_text}

Hãy chọn đúng 1 đáp án trong các lựa chọn trên.
Chỉ trả về JSON hợp lệ theo schema:
{{"selected_option": <số nguyên 1-5>, "answer_text": "<nội dung đáp án>", "reason": "<giải thích ngắn>"}}
"""

    return [
        {
            "role": "system",
            "content": (
                "Bạn là baseline LLM-only cho bài toán OntologyQA. "
                "Không sinh SPARQL, không gọi công cụ, không giả định có truy cập knowledge graph. "
                "Chỉ chọn đáp án từ các lựa chọn được cung cấp."
            ),
        },
        {"role": "user", "content": user_prompt},
    ]


messages = build_llm_only_messages(sample)
print(messages[1]["content"])

Câu hỏi tiếng Việt:
Nguyễn Hữu Cảnh có bao nhiêu người thân

Loại câu hỏi: counting

Các lựa chọn:
1. 1
2. 2
3. 3
4. 4
5. 5

Hãy chọn đúng 1 đáp án trong các lựa chọn trên.
Chỉ trả về JSON hợp lệ theo schema:
{"selected_option": <số nguyên 1-5>, "answer_text": "<nội dung đáp án>", "reason": "<giải thích ngắn>"}



## 4. Gọi OpenRouter

Cell này dùng `client.chat.completions.with_raw_response.create(...)` để lấy cả nội dung model lẫn HTTP response headers.

In [4]:
if not API_KEY:
    raise RuntimeError("Thiếu OPENROUTER_API_KEY. Hãy set biến môi trường trước khi chạy cell này.")

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=API_KEY,
)

started = time.perf_counter()
raw_response = client.chat.completions.with_raw_response.create(
    model=MODEL,
    messages=messages,
    temperature=0,
    max_tokens=256,
)
wall_time_s = time.perf_counter() - started

completion = raw_response.parse()
headers = dict(raw_response.headers)
content = completion.choices[0].message.content

print("Wall time (client-side seconds):", round(wall_time_s, 3))
print("Model response:")
print(content)
print("\nUsage:")
print(completion.usage)

interesting_headers = {
    key: value
    for key, value in headers.items()
    if any(token in key.lower() for token in ["openrouter", "provider", "latency", "processing", "time", "rate", "token"])
}
print("\nInteresting response headers:")
print(json.dumps(interesting_headers, ensure_ascii=False, indent=2))

Wall time (client-side seconds): 3.112
Model response:
```json
{"selected_option": 1, "answer_text": "1", "reason": "Dựa trên dữ liệu ontology thông thường về nhân vật lịch sử này, chỉ có một mối quan hệ người thân cụ thể được ghi nhận trong tập dữ liệu mẫu là cha của ông."}
```

Usage:
CompletionUsage(completion_tokens=65, prompt_tokens=178, total_tokens=243, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=4.914e-05, is_byok=False, cost_details={'upstream_inference_cost': 4.914e-05, 'upstream_inference_prompt_cost': 2.314e-05, 'upstream_inference_completions_cost': 2.6e-05})

Interesting response headers:
{}


## 5. Parse và chấm thử 1 mẫu

In [5]:
def parse_selected_option(text: str) -> int | None:
    cleaned = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", cleaned, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        cleaned = fenced.group(1).strip()

    try:
        payload = json.loads(cleaned)
        selected = payload.get("selected_option")
        return int(selected) if selected is not None else None
    except Exception:
        match = re.search(r"selected_option[^0-9]*(\d+)", cleaned)
        return int(match.group(1)) if match else None


predicted_option = parse_selected_option(content)
is_correct = predicted_option == sample["correct_option"]

result = {
    "question_number": sample["number"],
    "question": sample["question"],
    "gold_answer": sample["gold_answer"],
    "correct_option": sample["correct_option"],
    "predicted_option": predicted_option,
    "is_correct": is_correct,
}

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "question_number": 11,
  "question": "Nguyễn Hữu Cảnh có bao nhiêu người thân",
  "gold_answer": 3,
  "correct_option": 3,
  "predicted_option": 1,
  "is_correct": false
}
